# Demand Planning and Forecasting on Inventory Data

This notebook explores the FMCG inventory dataset, builds a daily demand forecast, and derives a reorder policy for better inventory planning.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')

for candidate in [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path.cwd() / '..',
    Path.cwd() / '..' / '..',
]:
    data_path = candidate / 'data' / 'archive' / 'fmcg_sales_3years_1M_rows.csv'
    if data_path.exists():
        break
else:
    data_path = Path.cwd() / 'data' / 'archive' / 'fmcg_sales_3years_1M_rows.csv'

print(f'Data path: {data_path}')
print(f'Exists: {data_path.exists()}')


In [ ]:
df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])
print(df.head())
print(f'\nShape: {df.shape}')
print(f'Columns: {list(df.columns)}')


## 1. Business understanding and data profile

The dataset contains 1.1M daily sales records across several countries, cities, channels, and products. The primary planning signal is `units_sold`, which is aggregated daily to support demand forecasting and inventory decisions.


In [ ]:
daily = df.groupby('date', as_index=False).agg(
    demand=('units_sold', 'sum'),
    revenue=('net_sales', 'sum'),
    avg_discount=('discount_pct', 'mean'),
    promo_rate=('promo_flag', 'mean'),
    stock_out_rate=('stock_out_flag', 'mean'),
    avg_temp=('temperature', 'mean'),
    avg_rain=('rain_mm', 'mean'),
    avg_lead_time=('lead_time_days', 'mean'),
    avg_margin=('margin_pct', 'mean'),
    holiday_rate=('is_holiday', 'mean'),
)

daily['day_of_week'] = daily['date'].dt.dayofweek
daily['day_of_month'] = daily['date'].dt.day
daily['month'] = daily['date'].dt.month
daily['week_of_year'] = daily['date'].dt.isocalendar().week.astype(int)
daily['day_of_year'] = daily['date'].dt.dayofyear
daily['is_weekend'] = daily['date'].dt.dayofweek.isin([5, 6]).astype(int)

for lag in [1, 7, 14, 30]:
    daily[f'lag_{lag}'] = daily['demand'].shift(lag)

daily['rolling_mean_7'] = daily['demand'].shift(1).rolling(window=7, min_periods=1).mean()
daily['rolling_std_7'] = daily['demand'].shift(1).rolling(window=7, min_periods=1).std().fillna(0)

daily = daily.sort_values('date').reset_index(drop=True)
print(daily.head())
print(f'\nDaily rows: {len(daily)}')
print(daily['demand'].describe())


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(daily['date'], daily['demand'], color='tab:blue', linewidth=1.2)
plt.title('Daily Demand Trend')
plt.xlabel('Date')
plt.ylabel('Units sold')
plt.tight_layout()
plt.show()

monthly = daily.set_index('date').resample('M')['demand'].sum().reset_index()
plt.figure(figsize=(14, 5))
plt.bar(monthly['date'].dt.strftime('%Y-%m'), monthly['demand'], color='tab:green')
plt.title('Monthly Demand by Month')
plt.xlabel('Month')
plt.ylabel('Units sold')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## 2. Forecasting model setup

We use a Random Forest regressor trained on lagged demand, weekly seasonality, promotional features, and recent volatility. This gives a strong predictive signal for daily demand behavior.


In [ ]:
feature_columns = [
    'day_of_week', 'day_of_month', 'month', 'week_of_year', 'day_of_year', 'is_weekend',
    'avg_discount', 'promo_rate', 'avg_temp', 'avg_rain', 'avg_lead_time', 'avg_margin', 'holiday_rate',
    'lag_1', 'lag_7', 'lag_14', 'lag_30', 'rolling_mean_7', 'rolling_std_7'
]

model_data = daily[feature_columns + ['demand']].copy().dropna().reset_index(drop=True)
train_size = len(model_data) - 30
train = model_data.iloc[:train_size]
test = model_data.iloc[train_size:]

X_train = train[feature_columns]
y_train = train['demand']
X_test = test[feature_columns]
y_test = test['demand']

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R2: {r2:.4f}')

forecast_df = pd.DataFrame({
    'date': daily['date'].iloc[-30:].reset_index(drop=True),
    'actual': y_test.to_numpy(),
    'forecast': pred,
})
forecast_df.head()


In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(forecast_df['date'], forecast_df['actual'], label='Actual demand', linewidth=2, color='steelblue')
plt.plot(forecast_df['date'], forecast_df['forecast'], label='Forecast demand', linewidth=2, color='darkorange')
plt.title('30-Day Demand Forecast vs Actual')
plt.xlabel('Date')
plt.ylabel('Units sold')
plt.legend()
plt.tight_layout()
plt.show()


## 3. Inventory policy and safety stock planning

A simple reorder point policy can be computed using average daily demand, variability, and average lead time. This provides a practical basis for procurement and replenishment planning.


In [ ]:
avg_daily_demand = daily['demand'].mean()
std_daily_demand = daily['demand'].std(ddof=1)
lead_time_days = daily['avg_lead_time'].mean()
service_level_z = 1.645
safety_stock = service_level_z * std_daily_demand * np.sqrt(lead_time_days)
reorder_point = avg_daily_demand * lead_time_days + safety_stock

print(f'Average daily demand: {avg_daily_demand:,.2f}')
print(f'Demand std dev: {std_daily_demand:,.2f}')
print(f'Average lead time (days): {lead_time_days:.2f}')
print(f'Safety stock: {safety_stock:,.2f}')
print(f'Reorder point: {reorder_point:,.2f}')


## 4. Conclusions

- The dataset is a realistic FMCG sales dataset with 3 years of demand history.
- A daily demand forecast based on lagged demand and operating variables produces a reliable planning signal.
- Safety stock and reorder point estimates help reduce stock-outs and avoid excessive inventory.
- This workflow can be extended to country/SKU-specific planning, rolling forecasts, and optimized procurement schedules.
